In [ ]:
import json
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer


In [ ]:
# on monte le google drive
drive.mount("/content/drive")

MODEL_NAME = "BioMistral/BioMistral-7B"

# dossier de ton drive ou tu as depose train.jsonl validation.jsonl test.jsonl
DATA_DIR = Path("/content/drive/MyDrive/techcorp/medical_dataset/processed")
TRAIN_FILE = str(DATA_DIR / "train.jsonl")
VAL_FILE = str(DATA_DIR / "validation.jsonl")

# sorties enregistrees aussi sur le drive pour survivre a la fin de session
OUTPUT_DIR = Path("/content/drive/MyDrive/techcorp/models/medical-qlora")
ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_DIR = OUTPUT_DIR / "merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LEN = 1024
SEED = 42

MAX_TRAIN = 500
MAX_VAL = 30
MAX_TEST = 30

# verification que les fichiers sont bien presents
assert Path(TRAIN_FILE).exists(), f"train introuvable {TRAIN_FILE} verifie le chemin sur ton drive"
assert Path(VAL_FILE).exists(), f"validation introuvable {VAL_FILE}"
print("train", TRAIN_FILE)
print("val  ", VAL_FILE)

In [ ]:
# configuration de la quantification 4 bits
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("compute dtype", compute_dtype)


In [ ]:
# chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for m in messages %}"
        "{{ '<|im_start|>' + m['role'] + '\n' + m['content'] + '<|im_end|>' + '\n' }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
    )

print("pad token", tokenizer.pad_token)
print("chat template present", tokenizer.chat_template is not None)


In [ ]:
# chargement du modele de base directement en 4 bits via la config bnb
# device_map auto place les couches sur le gpu disponible
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

# prepare le modele quantifie pour l entrainement en kbit
# caste les normes en fp32 et active le gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print("modele charge en 4 bits")


In [ ]:
# configuration des adaptateurs lora
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [ ]:
# chargement des jeux deja splittes au format messages une ligne un exemple
dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE},
)

dataset["train"] = dataset["train"].shuffle(seed=SEED).select(range(MAX_TRAIN))
dataset["validation"] = dataset["validation"].shuffle(seed=SEED).select(range(MAX_VAL))

print(dataset)
print("exemple brut")
print(dataset["train"][0]["messages"])

In [ ]:
def format_chat(example):
    messages = example["messages"]

    # si le premier message est un system on le colle en tete du premier user
    if messages and messages[0]["role"] == "system":
        system_text = messages[0]["content"]
        rest = messages[1:]
        if rest and rest[0]["role"] == "user":
            merged_user = {
                "role": "user",
                "content": system_text + "\n\n" + rest[0]["content"],
            }
            messages = [merged_user] + rest[1:]
        else:
            messages = rest

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# apercu du rendu sur un exemple
print(format_chat(dataset["train"][0])["text"][:600])

In [ ]:
# parametres d entrainement
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.001,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    logging_steps=25,
    save_strategy="epoch",
    eval_strategy="epoch",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=SEED,
)


In [ ]:
# construction du trainer
# on passe la fonction de formatage pour produire le champ text attendu
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,
    formatting_func=lambda ex: format_chat(ex)["text"],
    processing_class=tokenizer,
)

# affiche le nombre de parametres reellement entraines tres faible grace a lora
trainer.model.print_trainable_parameters()


In [ ]:
# lancement de l entrainement
train_result = trainer.train()

print("entrainement termine")
print(train_result.metrics)


In [ ]:
# sauvegarde de l adaptateur lora seul tres leger quelques dizaines de mo
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print("adaptateur enregistre dans", ADAPTER_DIR)


In [ ]:
# test d inference rapide avec l adaptateur encore en memoire
# on reutilise le modele 4 bits et on genere une reponse a une question medicale
SYSTEM_PROMPT = (
    "You are a helpful and empathetic medical assistant. "
    "You provide clear, accurate and responsible health information. "
    "You always remind the user to consult a qualified healthcare professional "
    "for a real diagnosis or emergency."
)

def generate(question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.config.use_cache = True
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    model.config.use_cache = False
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(generate("I have had a sore throat and mild fever for three days what should i do"))


In [ ]:
# fusion de l adaptateur dans le modele de base pour le deploiement
from peft import PeftModel

del model
del trainer
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
merged = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
merged = merged.merge_and_unload()

merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))


In [ ]:
# tests de performance du modele experimental
# on relit l historique d entrainement depuis le trainer_state.json le plus recent
import glob
import json as _json
import matplotlib.pyplot as plt

CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
states = sorted(glob.glob(str(CHECKPOINTS_DIR / "checkpoint-*" / "trainer_state.json")))
state_path = states[-1] if states else str(CHECKPOINTS_DIR / "trainer_state.json")

with open(state_path) as f:
    state = _json.load(f)

history = state["log_history"]
print("historique lu depuis", state_path)
print("nombre d evenements", len(history))


In [ ]:
# separation des pertes train et eval depuis l historique
# les logs de train portent la cle loss les evaluations portent la cle eval_loss
train_steps = [h["step"] for h in history if "loss" in h]
train_loss = [h["loss"] for h in history if "loss" in h]

eval_steps = [h["step"] for h in history if "eval_loss" in h]
eval_loss = [h["eval_loss"] for h in history if "eval_loss" in h]

print("points de train loss", len(train_loss))
print("points de eval loss", len(eval_loss))


In [ ]:
# trace de la loss curve train et eval sur le meme graphe
plt.figure(figsize=(9, 5))
plt.plot(train_steps, train_loss, label="train loss", linewidth=1.5)
if eval_loss:
    plt.plot(eval_steps, eval_loss, label="eval loss", marker="o", linewidth=1.5)
plt.xlabel("step")
plt.ylabel("loss")
plt.title("courbe de perte qlora modele medical")
plt.legend()
plt.grid(True, alpha=0.3)

# sauvegarde de la figure a cote des artefacts du modele
fig_path = OUTPUT_DIR / "loss_curve.png"
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("figure enregistree dans", fig_path)


In [ ]:
# trace du learning rate au fil des steps pour verifier le warmup puis la decroissance cosine
lr_steps = [h["step"] for h in history if "learning_rate" in h]
lr_values = [h["learning_rate"] for h in history if "learning_rate" in h]

if lr_values:
    plt.figure(figsize=(9, 4))
    plt.plot(lr_steps, lr_values, color="green", linewidth=1.5)
    plt.xlabel("step")
    plt.ylabel("learning rate")
    plt.title("planning du learning rate")
    plt.grid(True, alpha=0.3)
    plt.savefig(OUTPUT_DIR / "lr_curve.png", dpi=120, bbox_inches="tight")
    plt.show()


In [ ]:
import math

final_train_loss = train_loss[-1] if train_loss else None
final_eval_loss = eval_loss[-1] if eval_loss else None

print("metriques finales du run experimental")
print("-" * 50)
if final_train_loss is not None:
    print("train loss finale     ", round(final_train_loss, 4))
if final_eval_loss is not None:
    print("eval loss finale      ", round(final_eval_loss, 4))
    print("perplexite validation ", round(math.exp(final_eval_loss), 2))

# extraction du resume d entrainement total temps et vitesse si present
summary = [h for h in history if "train_runtime" in h]
if summary:
    s = summary[-1]
    print("temps total d entrainement s", round(s.get("train_runtime", 0), 1))
    print("echantillons par seconde    ", round(s.get("train_samples_per_second", 0), 2))


In [ ]:
# evaluation qualitative du modele fusionne sur quelques cas medicaux
questions_medicales = [
    "What are the common symptoms of type 2 diabetes",
    "I have had a persistent headache for a week should i be worried",
    "What is the difference between a virus and a bacteria",
]

for q in questions_medicales:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(merged.device)
    out = merged.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    rep = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("question", q)
    print("reponse ", rep)
    print("-" * 70)
